In [1]:
# data science imports
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import openml
from sklearn.preprocessing import StandardScaler

# file system
import os
from os.path import join as oj

In [2]:

def get_data(data_source, data_id):
    """
    Fetches dataset from either UCI Machine Learning Repository or OpenML.
    
    Parameters:
    data_source (str): The source of the dataset, either 'uci' or 'openml'.
    data_id (int): The ID of the dataset.
    
    Returns:
    X (np.ndarray): The feature matrix.
    y (np.ndarray): The target vector.
    """
    
    # ensure that data source is either 'uci' or 'openml'
    if data_source not in ["uci", "openml"]:
        raise ValueError("data_source must be either 'uci' or 'openml'")
    
    # handle case where data comes from uci
    if data_source == "uci":
        
        # get pandas df X and numpy array y
        dataset = fetch_ucirepo(id=data_id)
        X = dataset.data.features
        y = dataset.data.targets.to_numpy().flatten()
        
        # handle breast cancer dataset
        if data_id == 15:
            # remove rows with 'nan' entries for 'Bare_nuclei'
            X = X.dropna()
            # remove same observations from dataframe y
            y = y[X.index]
            # reset index
            X = X.reset_index(drop=True)
            # transform y from 2/4 to 0/1
            y = (y == 4).astype(int)
        
        X = X.to_numpy() # convert to numpy

    if data_source == "openml":
        
        # get data
        task = openml.tasks.get_task(data_id)
        dataset = task.get_dataset()
        X, y, categorical_mask, col_names = \
            dataset.get_data(target=dataset.default_target_attribute,
                            dataset_format="array")
        
    # center and scale the covariates
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # # sample 2000 rows of X and y if X has more than 2000 rows
    # if X.shape[0] > 2000:
    #     np.random.seed(42)
    #     indices = np.random.choice(X.shape[0], 2000, replace=False)
    #     X = X[indices]
    #     y = y[indices]

    return X, y

In [3]:
# data_ids = [361260, 361259, 361243, 361063, 361062, 361071]
# data_ids = [361254]
data_ids = [361260, 361254, 361259, 361253, 361243, 361242,
            361063, 361069, 361062, 9978, 361071, 43]
for id in data_ids:
    X, y = get_data("openml", id)
    data_dir = oj("data", str(id))
    os.makedirs(data_dir, exist_ok=True)
    np.savetxt(oj(data_dir, "X.csv"), X, delimiter=",")
    np.savetxt(oj(data_dir, "y.csv"), y, delimiter=",")

/tmp/ipykernel_288460/1720729341.py:6: FutureWarning: Starting from Version 0.15.0 `download_splits` will default to ``False`` instead of ``True`` and be independent from `download_data`. To disable this message until version 0.15 explicitly set `download_splits` to a bool.
  X, y = get_data("openml", id)
/scratch/users/zachrewolinski/conda/envs/mdi/lib/python3.10/site-packages/openml/tasks/functions.py:442: FutureWarning: Starting from Version 0.15 `download_data`, `download_qualities`, and `download_features_meta_data` will all be ``False`` instead of ``True`` by default to enable lazy loading. To disable this message until version 0.15 explicitly set `download_data`, `download_qualities`, and `download_features_meta_data` to a bool while calling `get_dataset`.
  dataset = get_dataset(task.dataset_id, *dataset_args, **get_dataset_kwargs)
/scratch/users/zachrewolinski/conda/envs/mdi/lib/python3.10/site-packages/openml/tasks/task.py:150: FutureWarning: Starting from Version 0.15 `downl